# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

In [1]:
# =============================================================================
# CELL 1 — Install & vLLM load (run this FIRST every session)
# =============================================================================
!pip install vllm==0.8.5 sympy==1.13.1 antlr4-python3-runtime==4.11.1 -q

import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"

from vllm import LLM, SamplingParams
print("vLLM loaded!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.26.0 which is incompatible.
google-adk 1.21.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.26.0 which is incompatible.
google-adk 1.21.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.26.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-api>=1.35.0, but you have opentelemetry-api 1.26.0 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.26.0 which is incompatible.
opentelemetry-exporter-gcp-monitoring 1.11.0a0 requires opentelemetry-api~=1.30, but you have opentelemetry-api 1.26.0 which is in

In [6]:
# =============================================================================
# CELL 2 — Mount Drive
# =============================================================================
# !fusermount -u /content/drive 2>/dev/null
# !rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# =============================================================================
# CELL 3 — Imports & Config
# =============================================================================
import re
import csv
import json
import sys
from pathlib import Path
from typing import Optional
from transformers import AutoTokenizer
from tqdm.auto import tqdm

MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH   = "/content/drive/MyDrive/competition_project/data/public.jsonl"
OUTPUT_PATH = "/content/drive/MyDrive/competition_project/results"

In [8]:
# =============================================================================
# CELL 4 — Load Dataset
# =============================================================================
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 1126 questions  (375 MCQ, 751 free-form)


In [9]:
# =============================================================================
# CELL 5 — Prompts
# =============================================================================
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician competing in a math olympiad. "
    "Think carefully and systematically, checking your work at each step. "
    "Simplify all expressions fully to their simplest numerical or symbolic form. "
    "The question uses [ANS] as placeholders for answers. "
    "Put your final answer inside \\boxed{}. "
    "For multiple sub-answers, use a single \\boxed{} with comma separation "
    "in the same order as the [ANS] placeholders, e.g. \\boxed{3, 7}. "
    "Always double-check your arithmetic before giving the final answer."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician competing in a math olympiad. "
    "Solve the problem completely first, then check which answer choice matches. "
    "Output ONLY the letter of the single best answer inside \\boxed{}, e.g. \\boxed{C}. "
    "Do not write anything after the boxed letter."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"({lbl}) {opt.strip()}" for lbl, opt in zip(labels, options))
        user = (
            f"{question}\n\n"
            f"Answer Choices:\n{opts_text}\n\n"
            "Which option is correct? Output only \\boxed{{<letter>}}."
        )
        return SYSTEM_PROMPT_MCQ, user
    return SYSTEM_PROMPT_MATH, question


In [7]:
# =============================================================================
# CELL 6 — Load saved responses (skip inference)
# =============================================================================
responses = {}
backup_path = Path(OUTPUT_PATH) / "all_responses.jsonl"
with open(backup_path) as f:
    for line in f:
        rec = json.loads(line)
        responses[rec["id"]] = rec["response"]
print(f"Loaded {len(responses)} saved responses.")

Loaded 1126 saved responses.


In [10]:
# =============================================================================
# CELL 7 — Score
# =============================================================================
!cp /content/drive/MyDrive/competition_project/judger.py /content/judger.py
!cp /content/drive/MyDrive/competition_project/utils.py /content/utils.py
sys.path.insert(0, "/content")
from judger import Judger
judger = Judger(strict_extract=False)
print("Judger loaded!")

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

# results = []
# for item in tqdm(data, desc="Scoring"):
#     response = responses.get(item.get("id"), "")
#     is_mcq = bool(item.get("options"))
#     gold   = item["answer"]

#     if is_mcq:
#         correct = score_mcq(response, str(gold))
#     else:
#         gold_list = gold if isinstance(gold, list) else [gold]
#         try:
#             correct = judger.auto_judge(
#                 pred=response,
#                 gold=gold_list,
#                 options=[[]] * len(gold_list),
#             )
#         except Exception:
#             correct = False

#     results.append({
#         "id":       item.get("id"),
#         "is_mcq":   is_mcq,
#         "gold":     gold,
#         "response": response,
#         "correct":  correct,
#     })

# print(f"Scoring complete. {len(results)} results.")

Judger loaded!


In [9]:
# =============================================================================
# CELL 8 — Summary
# =============================================================================
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :  182 /  375  (48.53%)
  Free-form  :  373 /  751  (49.67%)
  Overall    :  555 / 1126  (49.29%)


In [10]:
# =============================================================================
# CELL 9 — Save submission CSV
# =============================================================================
out_path = Path(OUTPUT_PATH) / "submission.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "response"])
    for item in data:
        resp = responses.get(item.get("id"), "")
        writer.writerow([item["id"], resp])

print(f"Saved {len(responses)} rows to {out_path}")

Saved 1126 rows to /content/drive/MyDrive/competition_project/results/submission.csv


In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    trust_remote_code=True,
    max_num_seqs=32,
    enforce_eager=True,
    disable_async_output_proc=True,
)
print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

INFO 06-01 00:07:44 [weight_utils.py:281] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 18.648834 seconds


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 06-01 00:07:46 [loader.py:458] Loading weights took 2.49 seconds
INFO 06-01 00:07:47 [model_runner.py:1140] Model loading took 7.6065 GiB and 22.010859 seconds
INFO 06-01 00:07:49 [worker.py:287] Memory profiling takes 1.29 seconds
INFO 06-01 00:07:49 [worker.py:287] the current vLLM instance can use total_gpu_memory (79.25GiB) x gpu_memory_utilization (0.90) = 71.33GiB
INFO 06-01 00:07:49 [worker.py:287] model weights take 7.61GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 0.61GiB; the rest of the memory reserved for KV Cache is 63.02GiB.
INFO 06-01 00:07:49 [executor_base.py:112] # cuda blocks: 28680, # CPU blocks: 1820
INFO 06-01 00:07:49 [executor_base.py:117] Maximum concurrency for 8192 tokens per request: 56.02x
INFO 06-01 00:07:51 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 4.58 seconds
Model loaded.


In [15]:
# private_data = [json.loads(line) for line in open("/content/drive/MyDrive/competition_project/data/private.jsonl")]
# print(f"Loaded {len(private_data)} private questions")

# prompts = []
# prompt_ids = []
# for item in private_data:
#     system, user = build_prompt(item["question"], item.get("options"))
#     prompt_text = tokenizer.apply_chat_template(
#         [{"role": "system", "content": system},
#          {"role": "user",   "content": user}],
#         tokenize=False,
#         add_generation_prompt=True,
#     )
#     prompts.append(prompt_text)
#     prompt_ids.append(item.get("id"))

# print(f"Generating responses for {len(prompts)} questions...")
# outputs = llm.generate(prompts, SamplingParams(max_tokens=8192, temperature=0.0))

# private_responses = {}
# for item_id, out in zip(prompt_ids, outputs):
#     private_responses[item_id] = out.outputs[0].text.strip()

# # Save backup
# backup_path = Path(OUTPUT_PATH) / "private_responses.jsonl"
# with open(backup_path, "w") as f:
#     for item_id, resp in private_responses.items():
#         f.write(json.dumps({"id": item_id, "response": resp}) + "\n")

# # Save submission CSV
# private_csv = Path(OUTPUT_PATH) / "private_submission.csv"
# with open(private_csv, "w", newline="") as f:
#     writer = csv.writer(f)
#     writer.writerow(["id", "response"])
#     for item in private_data:
#         resp = private_responses.get(item.get("id"), "")
#         writer.writerow([item["id"], resp])

# print(f"Saved private submission to {private_csv}")

In [16]:
!pip install peft trl bitsandbytes accelerate datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 65.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.21.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.26.0 which is incompatible.
google-adk 1.21.0 requires opentelemetry-exporter-otlp-proto-http>=1.36.0, but you have opentelemetry-exporter-otlp-proto-http 1.26.0 which is incompatible.
google-adk 1.21.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.26.0 which is incompatible.


In [3]:
# CELL B — Fine-tuning (FAST: GSM8K only, 1 epoch, 3000 samples)
import os
import json
import torch
from pathlib import Path
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_ID   = "Qwen/Qwen3-4B-Thinking-2507"
OUTPUT_DIR = "/content/drive/MyDrive/competition_project/qlora_model"  # save to Drive!
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 3
BATCH_SIZE = 8
GRAD_ACCUM = 2
LEARNING_RATE = 2e-4
MAX_TRAIN_SAMPLES = 3000

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def format_gsm8k(example):
    answer_text = example.get("answer", "")
    if "####" in answer_text:
        parts = answer_text.split("####")
        answer_text = f"{parts[0].strip()}\n\nThe answer is \\boxed{{{parts[1].strip()}}}"
    messages = [
        {"role": "system", "content": "You are an expert mathematician. Solve the problem step-by-step. Put your final answer inside \\boxed{}."},
        {"role": "user", "content": example.get("question", "")},
        {"role": "assistant", "content": answer_text},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

# Only GSM8K (smallest, fastest, answers reboxed to \boxed{})
all_datasets = []
print("Loading GSM8K...")
gsm = load_dataset("openai/gsm8k", "main", split="train")
gsm = gsm.map(format_gsm8k, remove_columns=gsm.column_names)
all_datasets.append(gsm)
print(f"  GSM8K: {len(gsm)} examples")

train_dataset = concatenate_datasets(all_datasets).shuffle(seed=42)
if len(train_dataset) > MAX_TRAIN_SAMPLES:
    train_dataset = train_dataset.select(range(MAX_TRAIN_SAMPLES))
print(f"\nTotal training examples: {len(train_dataset)}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=25,
    save_strategy="epoch",
    save_total_limit=1,
    bf16=True,
    max_length=MAX_SEQ_LEN,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="text",
    packing=True,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nTraining complete! Model saved to {OUTPUT_DIR}")

Loading GSM8K...


Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

  GSM8K: 7473 examples

Total training examples: 3000


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

trainable params: 132,120,576 || all params: 4,154,588,672 || trainable%: 3.1801


...

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Step,Training Loss
25,0.765500
50,0.454000
75,0.349500
100,0.296200
125,0.250000



Training complete! Model saved to /content/drive/MyDrive/competition_project/qlora_model


In [2]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL  = "Qwen/Qwen3-4B-Thinking-2507"
ADAPTER_DIR = "/content/drive/MyDrive/competition_project/qlora_model"
MERGED_DIR  = "/content/qlora_merged"   # LOCAL disk

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16,
    device_map="cpu", trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model = model.merge_and_unload()
model.save_pretrained(MERGED_DIR, safe_serialization=True)

tok = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
tok.save_pretrained(MERGED_DIR)
print("Merged ->", MERGED_DIR)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'lora_ga_config', 'peft_version', 'qalora_group_size', 'target_parameters', 'use_bdlora', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Merged -> /content/qlora_merged


INFO 06-01 06:31:19 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 06-01 06:31:19 [__init__.py:239] Automatically detected platform cuda.


ImportError: /usr/local/lib/python3.12/dist-packages/vllm/_C.abi3.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSsb

In [3]:
# CELL C — DO NOT RUN. Merge already complete, qlora_merged/ on Drive.
# import torch
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer

# BASE_MODEL  = "Qwen/Qwen3-4B-Thinking-2507"
# ADAPTER_DIR = "/content/drive/MyDrive/competition_project/qlora_model"
# MERGED_DIR  = "/content/drive/MyDrive/competition_project/qlora_merged"

# base = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL, torch_dtype=torch.bfloat16,
#     device_map="cpu", trust_remote_code=True,
# )
# model = PeftModel.from_pretrained(base, ADAPTER_DIR)
# model = model.merge_and_unload()
# model.save_pretrained(MERGED_DIR, safe_serialization=True)

# tok = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
# tok.save_pretrained(MERGED_DIR)
# print("Merged ->", MERGED_DIR)

In [11]:
#cell D
from vllm import LLM, SamplingParams

MERGED_DIR = "/content/drive/MyDrive/competition_project/qlora_merged"

tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MERGED_DIR,
    dtype="bfloat16",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    trust_remote_code=True,
    max_num_seqs=32,
    enforce_eager=True,
    disable_async_output_proc=True,
)

prompts, prompt_ids = [], []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
    )
    prompts.append(prompt_text)
    prompt_ids.append(item.get("id"))

outputs = llm.generate(prompts, SamplingParams(max_tokens=8192, temperature=0.0))

ft_responses = {pid: out.outputs[0].text.strip()
                for pid, out in zip(prompt_ids, outputs)}

# back up so you don't have to regenerate
import json
from pathlib import Path
with open(Path(OUTPUT_PATH) / "ft_public_responses.jsonl", "w") as f:
    for pid, resp in ft_responses.items():
        f.write(json.dumps({"id": pid, "response": resp}) + "\n")
print("Generated", len(ft_responses), "responses")

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/competition_project/qlora_merged'. Use `repo_type` argument if needed.

In [ ]:
#cell E
results = []
for item in tqdm(data, desc="Scoring FT"):
    response = ft_responses.get(item.get("id"), "")
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]
    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(pred=response, gold=gold_list,
                                        options=[[]] * len(gold_list))
        except Exception:
            correct = False
    results.append({"id": item.get("id"), "is_mcq": is_mcq,
                    "gold": gold, "response": response, "correct": correct})

mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]
def acc(s): return sum(r["correct"] for r in s)/len(s)*100 if s else 0.0
print(f"MCQ      : {acc(mcq_res):.2f}%")
print(f"Free-form: {acc(free_res):.2f}%")
print(f"OVERALL  : {acc(results):.2f}%   (baseline was 49.29%)")

In [4]:
!ls -la /content/drive/MyDrive/competition_project/qlora_merged/

total 7872016
drwxr-xr-x 2 root root       4096 Jun  1 04:43 .
drwxr-xr-x 4 root root       4096 Jun  1 04:43 ..
-rw-r--r-- 1 root root        707 Jun  1 04:43 added_tokens.json
-rw-r--r-- 1 root root        728 Jun  1 04:43 config.json
-rw-r--r-- 1 root root        214 Jun  1 04:43 generation_config.json
-rw-r--r-- 1 root root    1671853 Jun  1 04:43 merges.txt
-rw-r--r-- 1 root root 4967215360 Jun  1 04:43 model-00001-of-00002.safetensors
-rw-r--r-- 1 root root 3077766632 Jun  1 04:43 model-00002-of-00002.safetensors
-rw-r--r-- 1 root root      32819 Jun  1 04:43 model.safetensors.index.json
-rw-r--r-- 1 root root        610 Jun  1 04:43 special_tokens_map.json
-rw-r--r-- 1 root root       9634 Jun  1 04:43 tokenizer_config.json
-rw-r--r-- 1 root root   11422654 Jun  1 04:43 tokenizer.json
-rw-r--r-- 1 root root    2776833 Jun  1 04:43 vocab.json


In [17]:
!find / -name "model-00001-of-00002.safetensors" 2>/dev/null
!ls -la /content/ 2>/dev/null
!du -sh /content/* 2>/dev/null | sort -h | tail -5

total 76
drwxr-xr-x 1 root root  4096 Jun  1 05:01 .
drwxr-xr-x 1 root root  4096 Jun  1 01:48 ..
drwxr-xr-x 1 root root  4096 Dec  9 14:41 .config
drwx------ 5 root root  4096 Jun  1 05:00 drive
-rw------- 1 root root 39031 Jun  1 05:01 judger.py
drwxr-xr-x 2 root root  4096 Jun  1 05:01 __pycache__
drwxr-xr-x 1 root root  4096 Dec  9 14:42 sample_data
-rw------- 1 root root 11546 Jun  1 05:01 utils.py
12K	/content/utils.py
40K	/content/judger.py
60K	/content/__pycache__
55M	/content/sample_data
3.8G	/content/drive


In [5]:
import shutil, os
src = "/content/qlora_merged"
dst = "/content/drive/MyDrive/competition_project/qlora_merged"

# remove any stale partial copy on Drive first (only if it exists)
if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print("Copied. Files on Drive:", os.listdir(dst))

from google.colab import drive
drive.flush_and_unmount()
print("Flushed and unmounted")

Copied. Files on Drive: ['tokenizer_config.json', 'special_tokens_map.json', 'model-00001-of-00002.safetensors', 'merges.txt', 'model-00002-of-00002.safetensors', 'config.json', 'model.safetensors.index.json', 'vocab.json', 'tokenizer.json', 'added_tokens.json', 'generation_config.json']
Drive not mounted, so nothing to flush and unmount.
Flushed and unmounted


In [7]:
import os
d = "/content/drive/MyDrive/competition_project/qlora_merged"
for f in ["model-00001-of-00002.safetensors", "model-00002-of-00002.safetensors"]:
    p = os.path.join(d, f)
    print(f, os.path.getsize(p) if os.path.exists(p) else "MISSING")

model-00001-of-00002.safetensors 4967215360
model-00002-of-00002.safetensors 3077766632


In [8]:
import os
print(os.path.exists("/content/drive/MyDrive/competition_project/qlora_merged/model-00001-of-00002.safetensors"))
print(os.listdir("/content/drive/MyDrive/competition_project"))

True
['qlora_merged', 'qlora_model']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')